# Exact Duplicate Condition Audit

This notebook inspects the worst exact duplicate duplex groups and separates them into:
- same duplex + same conditions + conflicting labels
- same duplex + different conditions

The goal is to identify true candidate-removal groups versus biologically plausible repeats across changing conditions.

## Why This Audit Matters

- If the exact same duplex appears multiple times under the same conditions with very different inhibition values, that is a strong data-quality warning.
- If the same duplex appears under different concentrations, cell types, or times, then large label differences may be biologically real and should not be removed automatically.
- This notebook turns the previous duplicate warning signal into a cleaner action list.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.merge_historic_data import load_merged_dataset
from utils.pipeline import SiRNADataPipeline
from utils.splitter import GroupKFoldLeakPerGroup

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## Step 1: Rebuild The Prediction Frame

Why: this keeps the duplicate audit aligned with the same cleaned dataset and held-out prediction frame used in the hotspot analysis notebooks.

In [2]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

raw_df = load_merged_dataset(cmsirna_path, historic_path)
pipeline = SiRNADataPipeline(target_len=25, fetch_missing_mrna=True)
enriched_df = pipeline.enrich_dataset_with_encodings(
    raw_df,
    strict_cleaning=True,
    add_mrna=True,
)
X, groups, y = pipeline.prepare_for_classical_ml(
    enriched_df,
    target_column="Inhibition",
    use_normalized_conditions=False,
)

mask = ~np.isnan(y)
X = X[mask]
groups = groups[mask]
y = y[mask]
analysis_df = enriched_df.loc[mask].reset_index(drop=True).copy()

analysis_df["patent_group"] = analysis_df.get(
    "patent_ID", pd.Series(index=analysis_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")

analysis_df.shape

loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

(35444, 40)

In [3]:
frozen_params = {
    "n_estimators": 800,
    "max_depth": 4,
    "learning_rate": 0.15881823130907038,
    "subsample": 0.8812898741586134,
    "colsample_bytree": 0.7824379872752019,
    "min_child_weight": 4,
    "reg_lambda": 0.8342807691178866,
    "reg_alpha": 1.4296995092035882,
    "gamma": 0.07531958697602548,
}

gene_cv = GroupKFoldLeakPerGroup(n_splits=3, leak_n=0, random_state=42)
oof_frames = []

for fold_id, (train_idx, test_idx) in enumerate(gene_cv.split(X, y, groups), start=1):
    model = XGBRegressor(tree_method="hist", n_jobs=-1, random_state=42, **frozen_params)
    model.fit(X[train_idx], y[train_idx])
    fold_pred = model.predict(X[test_idx])

    fold_frame = analysis_df.iloc[test_idx].reset_index().rename(columns={"index": "source_index"}).copy()
    fold_frame["row_index"] = test_idx
    fold_frame["fold_id"] = fold_id
    fold_frame["group"] = groups[test_idx]
    fold_frame["y_true"] = y[test_idx]
    fold_frame["y_pred"] = fold_pred
    fold_frame["abs_error"] = (fold_frame["y_true"] - fold_frame["y_pred"]).abs()
    oof_frames.append(fold_frame)

predictions_df = pd.concat(oof_frames, ignore_index=True)
predictions_df.head()

,source_index,ID,patent_ID,Authorization_status,Accession_number,gene_target_symbol_name,Gene_ID,The_name_of_double_helix,Antisense_seqence,length_anti_sense_strand,Modifications_AntiSense_strand,Modification_Types_Antisense_strand,Sense_seqence,length_sense_strand,Modification_Types_Sense_strand,Inhibition,SD,Cell_Type,Concentration,Time_of_administration,Title,mRNA,Concentration_nM,Time_of_administration_h,mRNA_five_prime,mRNA_three_prime,edit_distance,target_site_pct,Sense_Sequence_One_Hot,Antisense_Sequence_One_Hot,Sense_Acid_One_Hot,Sense_Sugar_One_Hot,Sense_Linker_One_Hot,Antisense_Acid_One_Hot,Antisense_Sugar_One_Hot,Antisense_Linker_One_Hot,Cell_Type_One_Hot,Concentration_log10_nM,Concentration_norm,Time_norm,patent_group,row_index,fold_id,group,y_true,y_pred,abs_error
0,4680,004-03-02-01275-1n-XXX--1.20,CN108368506A,Substantive Examination,NM_005577.4,LPA,4018.0,AD00571,TCGGCAGUCCCUUCUGCGUTT,21.0,dTCfgGfcAfgUfcCfcUfuCfuGfcGfudTsdT,1*2'-Deoxy thymidine || 2*2'-Fluorocytidine ||...,ACGCAGAAGGGACUGCCGAT,20.0,1*2'-Fluoroadenosine || 2*2'-O-Methylcytidine ...,-1.2,14.6,Hep3B,1.0,24.0,用于抑制LPA的基因表达的组合物和方法,GUAAGUCAACAAUGUCCUGGGAUUGGGACACACUUUCUGGGCACUG...,1.0,24.0,GUAAGUCAACAAUGUCCUGGGAUUGGGACACACUUUCUGGGCACUG...,UUGGACGGGAGACAGAGUGAAGCAUCAACCUACUUAGAAGCUGAAA...,2.0,0.058778,"[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0....","[[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...",0.0,0.620954,0.121951,CN108368506A,4680,1,LPA,-1.2,34.769035,35.969035
1,4681,004-03-02-01276-1n-XXX-22.40,CN108368506A,Substantive Examination,NM_005577.4,LPA,4018.0,AD00572,TGUAGCACUCCUGCACCCCTT,21.0,dTGfuAfgCfaCfuCfcUfgCfaCfcCfcdTsdT,1*2'-Deoxy thymidine || 2*2'-Fluoroguanosine |...,GGGGUGCAGGAGUGCUACAT,20.0,1*2'-Fluoroguanosine || 2*2'-O-Methylguanosine...,22.4,6.2,Hep3B,1.0,24.0,用于抑制LPA的基因表达的组合物和方法,GUAAGUCAACAAUGUCCUGGGAUUGGGACACACUUUCUGGGCACUG...,1.0,24.0,GUAAGUCAACAAUGUCCUGGGAUUGGGACACACUUUCUGGGCACUG...,UUGGACGGGAGACAGAGUGAAGCAUCAACCUACUUAGAAGCUGAAA...,3.0,0.073084,"[[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0....","[[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...",0.0,0.620954,0.121951,CN108368506A,4681,1,LPA,22.4,0.201030,22.198970
2,4682,004-03-02-01277-1n-XXX-29.20,CN108368506A,Substantive Examination,NM_005577.4,LPA,4018.0,AD00573,TAAUAAGGGGCUGCCACAGTT,21.0,dTAfaUfaAfgGfgGfcUfgCfcAfcAfgdTsdT,1*2'-Deoxy thymidine || 2*2'-Fluoroadenosine |...,CUGUGGCAGCCCCUUAUUAT,20.0,1*2'-Fluorocytidine || 2*2'-O-Methyluridine ||...,29.2,5.4,Hep3B,1.0,24.0,用于抑制LPA的基因表达的组合物和方法,GUAAGUCAACAAUGUCCUGGGAUUGGGACACACUUUCUGGGCACUG...,1.0,24.0,GUAAGUCAACAAUGUCCUGGGAUUGGGACACACUUUCUGGGCACUG...,UUGGACGGGAGACAGAGUGAAGCAUCAACCUACUUAGAAGCUGAAA...,3.0,0.419219,"[[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 1....","[[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [1.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...",0.0,0.620954,0.121951,CN108368506A,4682,1,LPA,29.2,47.724365,18.524365
3,4683,004-03-02-01278-1n-XXX-55.90,CN108368506A,Substantive Examinatio

## Step 2: Build Exact Duplex Keys And Condition Keys

Why: we need one key for exact sequence identity and another key for whether the rows were measured under the same conditions.

In [4]:
def normalize_seq(value):
    if pd.isna(value):
        return None
    return "".join(str(value).upper().split())

predictions_df["sense_seq_norm"] = predictions_df["Sense_seqence"].map(normalize_seq)
predictions_df["antisense_seq_norm"] = predictions_df["Antisense_seqence"].map(normalize_seq)
predictions_df["duplex_key"] = (
    predictions_df["sense_seq_norm"].fillna("MISSING")
    + "||"
    + predictions_df["antisense_seq_norm"].fillna("MISSING")
)

condition_cols = [
    "group",
    "Cell_Type",
    "Concentration_nM",
    "Time_of_administration_h",
]

predictions_df["condition_key"] = predictions_df[condition_cols].astype(str).agg(" || ".join, axis=1)
predictions_df[["duplex_key", "condition_key"]].head()

,duplex_key,condition_key
0,ACGCAGAAGGGACUGCCGAT||TCGGCAGUCCCUUCUGCGUTT,LPA || Hep3B || 1.0 || 24.0
1,GGGGUGCAGGAGUGCUACAT||TGUAGCACUCCUGCACCCCTT,LPA || Hep3B || 1.0 || 24.0
2,CUGUGGCAGCCCCUUAUUAT||TAAUAAGGGGCUGCCACAGTT,LPA || Hep3B || 1.0 || 24.0
3,GGCAGCCCCUUAUUGUUAAT||TUAACAAUAAGGGGCUGCCTT,LPA || Hep3B || 1.0 || 24.0
4,AGCCCCUUAUUGUUAUACAT||TGUAUAACAAUAAGGGGCUTT,LPA || Hep3B || 1.0 || 24.0


## Step 3: Audit Exact Duplicate Groups

Why: we want to summarize each exact duplicate duplex group, then classify it as same-condition or different-condition.

In [5]:
rows = []
for duplex_key, dup_df in predictions_df.groupby("duplex_key"):
    if len(dup_df) < 2:
        continue

    unique_conditions = dup_df["condition_key"].nunique()
    rows.append({
        "duplex_key": duplex_key,
        "n_rows": len(dup_df),
        "n_condition_keys": unique_conditions,
        "same_condition_only": unique_conditions == 1,
        "group": dup_df["group"].mode().iloc[0],
        "cell_types": sorted(dup_df["Cell_Type"].astype(str).unique().tolist()),
        "concentrations": sorted(dup_df["Concentration_nM"].astype(float).unique().tolist()),
        "times": sorted(dup_df["Time_of_administration_h"].astype(float).unique().tolist()),
        "patents": sorted(dup_df["patent_group"].astype(str).unique().tolist()),
        "y_true_min": float(dup_df["y_true"].min()),
        "y_true_max": float(dup_df["y_true"].max()),
        "y_true_range": float(dup_df["y_true"].max() - dup_df["y_true"].min()),
        "mean_y_true": float(dup_df["y_true"].mean()),
        "row_indices": dup_df["row_index"].tolist(),
        "ids": dup_df.get("ID", pd.Series(index=dup_df.index, dtype=object)).astype(str).tolist(),
    })

duplicate_audit = pd.DataFrame(rows)
duplicate_audit.sort_values(["y_true_range", "n_rows"], ascending=[False, False]).head(30)

,duplex_key,n_rows,n_condition_keys,same_condition_only,group,cell_types,concentrations,times,patents,y_true_min,y_true_max,y_true_range,mean_y_true,row_indices,ids
6832,UUUUAUGUGCACACAUUAGGA||UCCUAAUGUGUGCACAUAAAACA,6,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-97.02,97.18,194.20,51.001667,"[16050, 16140, 16226, 16306, 16396, 16486]","[020-05-14-07142-10n-24h-71.82, 020-05-14-0714..."
6526,UUCCUGAUCACUAUGCAUUUA||UAAAUGCAUAGUGAUCAGGAAAG,12,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-91.48,99.68,191.16,64.055833,"[16013, 16056, 16101, 16146, 16191, 16232, 162...","[020-05-14-07104-10n-24h-87.15, 020-05-14-0714..."
4961,GUCUCACUUUCCAGCAAAA||UUUUGCUGGAAAGUGAGACCC,9,3,False,AGT,[Hep3B],"[0.01, 0.1, 1.0]",[24.0],[CN117448322A],-99.50,88.20,187.70,24.100000,"[29184, 29185, 29189, 29231, 29232, 29236, 292...","[078-12-02-13790-0.1n-24h-88.20, 078-12-02-137..."
4960,GUCUCACUUUCCAGCAAAAUU||UUUUGCUGGAAAGUGAGACCC,12,3,False,AGT,[Hep3B],"[0.01, 0.1, 1.0]",[24.0],[CN117448322A],-86.20,90.50,176.70,28.016667,"[29181, 29183, 29187, 29191, 29228, 29230, 292...","[078-12-02-13787-0.1n-24h-90.50, 078-12-02-137..."
4947,GUCGUUGGUGAAGUUUUUCAU||AUGAAAAACUUCACCAACGACUC,3,3,False,HSD17B13,"[COS7, Primary Cynomolgus Monkey Hepatocytes, ...",[50.0],[48.0],[CN112020556A],-95.00,80.60,175.60,13.000000,"[17822, 18076, 18330]","[032-09-09-08456-50n-48h-80.60, 032-09-03-0845..."
4502,GGAUGAUUGUACAGAAUCAUA||UAUGAUTCUGUACAAUCAUCCUG,6,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-80.61,90.96,171.57,42.060000,"[16086, 16176, 16254, 16342, 16432, 16522]","[020-05-14-07178-10n-24h-64.72, 020-05-14-0717..."
5492,UCAAACUAGUGCAUGAAUAGA||UCUAUUCAUGCACUAGUUUGAUA,11,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-74.04,97.28,171.32,39.504545,"[16027, 16070, 16115, 16160, 16205, 16281, 163...","[020-05-14-07118-10n-24h-71.30, 020-05-14-0716..."
5478,UAUUGAGAAUGAAGUGUGCAA||UUGCACACUUCAUUCUCAAUAAC,4,3,False,LPA,"[Huh7, RT-4]","[0.1, 1.0, 50.0]","[24.0, 48.0]",[CN116801886A],-91.64,78.57,170.21,31.347500,"[5651, 5742, 5830, 5891]","[009-03-05-03243-1n-XXX-78.57, 009-03-05-03243..."
6699,UUGUGCCUGUUUUAUGUGCAA||UUGCACAUAAAACAGGCACAAAG,12,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-69.52,98.48,168.00,61.400000,"[16034, 16077, 16122, 16167, 16212, 16248, 162...","[020-05-14-07125-10n-24h-72.77, 020-05-14-0716..."
4900,GUCCAAAGACGAAGUCGUGGU||ACCACGACUUCGUCUUUGGACCG,5,5,False,PNPLA3,"[Hep3B, HepG2]","[1.0, 10.0, 50.0]",[24.0],[WO2022256395A1],-86.36,80.80,167.16,-35.168000,"[11251, 11595, 11940, 12627, 12972]","[013-04-02-04394-50n-24h-80.80, 013-04-02-0439..."


## Step 4: Split Into Same-Condition Conflicts And Different-Condition Repeats

Why: this is the actual decision boundary for cleanup. Same-condition conflicts are much stronger removal candidates than different-condition repeats.

In [6]:
same_condition_conflicts = duplicate_audit.loc[
    duplicate_audit["same_condition_only"]
].sort_values(["y_true_range", "n_rows"], ascending=[False, False]).reset_index(drop=True)

different_condition_duplicates = duplicate_audit.loc[
    ~duplicate_audit["same_condition_only"]
].sort_values(["y_true_range", "n_rows"], ascending=[False, False]).reset_index(drop=True)

print("Same duplex + same conditions + conflicting labels")
display(same_condition_conflicts.head(50))

print("Same duplex + different conditions")
display(different_condition_duplicates.head(50))

Same duplex + same conditions + conflicting labels


,duplex_key,n_rows,n_condition_keys,same_condition_only,group,cell_types,concentrations,times,patents,y_true_min,y_true_max,y_true_range,mean_y_true,row_indices,ids
0,GGCAACUUCCGGGACGAUGTT||CAUCGUCCCGGAAGUUGCC,2,1,True,PCSK9,[HepG2],[30.0],[24.0],[CN101484588B],-98.00,0.00,98.00,-49.000000,"[29589, 29590]","[080-13-11-14077-30n-XXX-0.00, 080-13-11-14078..."
1,UGUGCUAGUCGCUGCAAAACUUU||AAGUUUUGCAGCGACUAGCACA,5,1,True,AGT,[HepG2],[10.0],[24.0],[WO2023056446A1],7.00,87.00,80.00,55.200000,"[27986, 28034, 28036, 28038, 28043]","[070-12-11-13033-10n-XXX-12.00, 070-12-11-1307..."
2,GGGAUACCUCACCAAGAUCTT||GAUCUUGGUGAGGUAUCCC,2,1,True,PCSK9,[HepG2],[30.0],[24.0],[CN101484588B],-40.00,40.00,80.00,0.000000,"[29371, 29372]","[080-13-11-13889-30n-XXX-40.00, 080-13-11-1389..."
3,UUAUUCGGAAAAGCCAGCUTT||AGCUGGCUUUUCCGAAUAA,2,1,True,PCSK9,[HepG2],[30.0],[24.0],[CN101484588B],-57.00,19.00,76.00,-19.000000,"[29555, 29556]","[080-13-11-14047-30n-XXX--57.00, 080-13-11-140..."
4,AUCGAGGGCAGGGUCAUGGTT||CCAUGACCCUGCCCUCGAU,2,1,True,PCSK9,[HepG2],[30.0],[24.0],[CN101484588B],-60.00,8.00,68.00,-26.000000,"[29499, 29500]","[080-13-11-13999-30n-XXX--60.00, 080-13-11-140..."
5,UUUCUGGAUGGCAUCUAGCTT||GCUAGAUGCCAUCCAGAAA,2,1,True,PCSK9,[HepG2],[30.0],[24.0],[CN101484588B],-23.00,44.00,67.00,10.500000,"[29819, 29820]","[080-13-11-14295-30n-XXX--23.00, 080-13-11-142..."
6,GAGGGUGUCUACGCCAUUGTT||CAAUGGCGUAGACACCCUC,2,1,True,PCSK9,[HepG2],[30.0],[24.0],[CN101484588B],0.00,64.00,64.00,32.000000,"[29644, 29645]","[080-13-11-14135-30n-XXX-64.00, 080-13-11-1413..."
7,CCCAUGUCGACUACAUCGATT||UCGAUGUAGUCGACAUGGG,2,1,True,PCSK9,[HepG2],[30.0],[24.0],[CN101484588B],-4.00,58.00,62.00,27.000000,"[29379, 29380]","[080-13-11-13895-30n-XXX-58.00, 080-13-11-1389..."
8,UCACCGUAUAUAUGGCAUGCAUU||AUGCAUGCCAUAUAUACGGUGA,5,1,True,AGT,[HepG2],[10.0],[24.0],[WO2023056446A1],22.00,83.00,61.00,50.200000,"[27955, 28006, 28014, 28022, 28031]","[070-12-11-13003-10n-XXX-22.00, 070-12-11-1304..."
9,UGUGGUCCUCCCACGCUCUCUAU||UAGAGAGCGUGGGAGGACCACA,4,1,True,AGT,[HepG2],[10.0],[24.0],[WO2023056446A1],22.00,83.00,61.00,48.000000,"[27983, 28009, 28017, 28025]","[070-12-11-13030-10n-XXX-83.00, 070-12-11-1304..."


Same duplex + different conditions


,duplex_key,n_rows,n_condition_keys,same_condition_only,group,cell_types,concentrations,times,patents,y_true_min,y_true_max,y_true_range,mean_y_true,row_indices,ids
0,UUUUAUGUGCACACAUUAGGA||UCCUAAUGUGUGCACAUAAAACA,6,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-97.02,97.18,194.20,51.001667,"[16050, 16140, 16226, 16306, 16396, 16486]","[020-05-14-07142-10n-24h-71.82, 020-05-14-0714..."
1,UUCCUGAUCACUAUGCAUUUA||UAAAUGCAUAGUGAUCAGGAAAG,12,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-91.48,99.68,191.16,64.055833,"[16013, 16056, 16101, 16146, 16191, 16232, 162...","[020-05-14-07104-10n-24h-87.15, 020-05-14-0714..."
2,GUCUCACUUUCCAGCAAAA||UUUUGCUGGAAAGUGAGACCC,9,3,False,AGT,[Hep3B],"[0.01, 0.1, 1.0]",[24.0],[CN117448322A],-99.50,88.20,187.70,24.100000,"[29184, 29185, 29189, 29231, 29232, 29236, 292...","[078-12-02-13790-0.1n-24h-88.20, 078-12-02-137..."
3,GUCUCACUUUCCAGCAAAAUU||UUUUGCUGGAAAGUGAGACCC,12,3,False,AGT,[Hep3B],"[0.01, 0.1, 1.0]",[24.0],[CN117448322A],-86.20,90.50,176.70,28.016667,"[29181, 29183, 29187, 29191, 29228, 29230, 292...","[078-12-02-13787-0.1n-24h-90.50, 078-12-02-137..."
4,GUCGUUGGUGAAGUUUUUCAU||AUGAAAAACUUCACCAACGACUC,3,3,False,HSD17B13,"[COS7, Primary Cynomolgus Monkey Hepatocytes, ...",[50.0],[48.0],[CN112020556A],-95.00,80.60,175.60,13.000000,"[17822, 18076, 18330]","[032-09-09-08456-50n-48h-80.60, 032-09-03-0845..."
5,GGAUGAUUGUACAGAAUCAUA||UAUGAUTCUGUACAAUCAUCCUG,6,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-80.61,90.96,171.57,42.060000,"[16086, 16176, 16254, 16342, 16432, 16522]","[020-05-14-07178-10n-24h-64.72, 020-05-14-0717..."
6,UCAAACUAGUGCAUGAAUAGA||UCUAUUCAUGCACUAGUUUGAUA,11,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-74.04,97.28,171.32,39.504545,"[16027, 16070, 16115, 16160, 16205, 16281, 163...","[020-05-14-07118-10n-24h-71.30, 020-05-14-0716..."
7,UAUUGAGAAUGAAGUGUGCAA||UUGCACACUUCAUUCUCAAUAAC,4,3,False,LPA,"[Huh7, RT-4]","[0.1, 1.0, 50.0]","[24.0, 48.0]",[CN116801886A],-91.64,78.57,170.21,31.347500,"[5651, 5742, 5830, 5891]","[009-03-05-03243-1n-XXX-78.57, 009-03-05-03243..."
8,UUGUGCCUGUUUUAUGUGCAA||UUGCACAUAAAACAGGCACAAAG,12,6,False,APP,"[Be(2)C cell line, Neuro2A cell line]","[0.1, 1.0, 10.0]",[24.0],[WO2020132227A2],-69.52,98.48,168.00,61.400000,"[16034, 16077, 16122, 16167, 16212, 16248, 162...","[020-05-14-07125-10n-24h-72.77, 020-05-14-0716..."
9,GUCCAAAGACGAAGUCGUGGU||ACCACGACUUCGUCUUUGGACCG,5,5,False,PNPLA3,"[Hep3B, HepG2]","[1.0, 10.0, 50.0]",[24.0],[WO2022256395A1],-86.36,80.80,167.16,-35.168000,"[11251, 11595, 11940, 12627, 12972]","[013-04-02-04394-50n-24h-80.80, 013-04-02-0439..."


## Step 5: Build A Conservative Candidate-Removal Table

Why: a practical first pass is to flag only same-condition duplicates with very large label ranges.

In [7]:
RANGE_THRESHOLD = 40.0

candidate_removal_table = same_condition_conflicts.loc[
    same_condition_conflicts["y_true_range"] >= RANGE_THRESHOLD
].copy()

candidate_removal_table[[
    "duplex_key",
    "group",
    "n_rows",
    "y_true_min",
    "y_true_max",
    "y_true_range",
    "cell_types",
    "concentrations",
    "times",
    "patents",
]].head(100)

,duplex_key,group,n_rows,y_true_min,y_true_max,y_true_range,cell_types,concentrations,times,patents
0,GGCAACUUCCGGGACGAUGTT||CAUCGUCCCGGAAGUUGCC,PCSK9,2,-98.0,0.0,98.0,[HepG2],[30.0],[24.0],[CN101484588B]
1,UGUGCUAGUCGCUGCAAAACUUU||AAGUUUUGCAGCGACUAGCACA,AGT,5,7.0,87.0,80.0,[HepG2],[10.0],[24.0],[WO2023056446A1]
2,GGGAUACCUCACCAAGAUCTT||GAUCUUGGUGAGGUAUCCC,PCSK9,2,-40.0,40.0,80.0,[HepG2],[30.0],[24.0],[CN101484588B]
3,UUAUUCGGAAAAGCCAGCUTT||AGCUGGCUUUUCCGAAUAA,PCSK9,2,-57.0,19.0,76.0,[HepG2],[30.0],[24.0],[CN101484588B]
4,AUCGAGGGCAGGGUCAUGGTT||CCAUGACCCUGCCCUCGAU,PCSK9,2,-60.0,8.0,68.0,[HepG2],[30.0],[24.0],[CN101484588B]
5,UUUCUGGAUGGCAUCUAGCTT||GCUAGAUGCCAUCCAGAAA,PCSK9,2,-23.0,44.0,67.0,[HepG2],[30.0],[24.0],[CN101484588B]
6,GAGGGUGUCUACGCCAUUGTT||CAAUGGCGUAGACACCCUC,PCSK9,2,0.0,64.0,64.0,[HepG2],[30.0],[24.0],[CN101484588B]
7,CCCAUGUCGACUACAUCGATT||UCGAUGUAGUCGACAUGGG,PCSK9,2,-4.0,58.0,62.0,[HepG2],[30.0],[24.0],[CN101484588B]
8,UCACCGUAUAUAUGGCAUGCAUU||AUGCAUGCCAUAUAUACGGUGA,AGT,5,22.0,83.0,61.0,[HepG2],[10.0],[24.0],[WO2023056446A1]
9,UGUGGUCCUCCCACGCUCUCUAU||UAGAGAGCGUGGGAGGACCACA,AGT,4,22.0,83.0,61.0,[HepG2],[10.0],[24.0],[WO2023056446A1]


## Step 6: Inspect One Duplicate Group In Detail

Why: before removing anything, it helps to inspect a few of the worst groups row by row.

In [8]:
if candidate_removal_table.empty:
    print("No candidate removal groups found at the current threshold.")
else:
    chosen_duplex = candidate_removal_table.iloc[0]["duplex_key"]
    print("Inspecting duplex:", chosen_duplex)
    display(
        predictions_df.loc[predictions_df["duplex_key"] == chosen_duplex, [
            "ID",
            "group",
            "Cell_Type",
            "Concentration_nM",
            "Time_of_administration_h",
            "patent_group",
            "y_true",
            "y_pred",
            "abs_error",
            "sense_seq_norm",
            "antisense_seq_norm",
        ]].sort_values("y_true")
    )

Inspecting duplex: GGCAACUUCCGGGACGAUGTT||CAUCGUCCCGGAAGUUGCC


,ID,group,Cell_Type,Concentration_nM,Time_of_administration_h,patent_group,y_true,y_pred,abs_error,sense_seq_norm,antisense_seq_norm
20436,080-13-11-14078-30n-XXX--98.00,PCSK9,HepG2,30.0,24.0,CN101484588B,-98.0,20.089787,118.089787,GGCAACUUCCGGGACGAUGTT,CAUCGUCCCGGAAGUUGCC
20435,080-13-11-14077-30n-XXX-0.00,PCSK9,HepG2,30.0,24.0,CN101484588B,0.0,60.210617,60.210617,GGCAACUUCCGGGACGAUGTT,CAUCGUCCCGGAAGUUGCC


## How To Use The Results

- `same_condition_conflicts` is the main cleanup table.
- `different_condition_duplicates` should usually be kept unless there is another quality problem.
- `candidate_removal_table` is a conservative shortlist for targeted cleaning.
- After review, use this shortlist for a cleaned-data ablation rather than deleting broad patents or whole genes.